<a href="https://colab.research.google.com/github/DoDaDrew18/CSE3104FinalProject/blob/main/Chipotle_data_merged.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# andrew comment: make sure to add instructions for download: pip install kagglehub in the terminal first before running
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jeffreybraun/chipotle-locations")

print("Path to dataset files:", path)

Path to dataset files: /Users/aviado/.cache/kagglehub/datasets/jeffreybraun/chipotle-locations/versions/3


In [24]:
import pandas as pd
import os

# List the contents of the directory to find the CSV file
file_list = os.listdir(path)

# Assuming there's only one CSV file in the directory, or it's named predictably
# We can filter for .csv files
csv_files = [f for f in file_list if f.endswith('.csv')]

# Construct the full path to the CSV file
csv_file_path = os.path.join(path, csv_files[0])
df = pd.read_csv(csv_file_path)




In [25]:
df.head()
df.dtypes

state         object
location      object
address       object
latitude     float64
longitude    float64
dtype: object

In [26]:
#extract new feature: zipcode
import re

df["zipcode"]= df["address"].str.extract(r'(\d{5}(?:-\d{4})?)\s*US$')
df.head()

,state,location,address,latitude,longitude,zipcode
0,Alabama,Auburn,"346 W Magnolia Ave Auburn, AL 36832 US",32.606813,-85.487328,36832
1,Alabama,Birmingham,"300 20th St S Birmingham, AL 35233 US",33.509721,-86.802756,35233
2,Alabama,Birmingham,"3220 Morrow Rd Birmingham, AL 35235 US",33.595581,-86.647437,35235
3,Alabama,Birmingham,"4719 Highway 280 Birmingham, AL 35242 US",33.422582,-86.698279,35242
4,Alabama,Cullman,"1821 Cherokee Ave SW Cullman, AL 35055 US",34.154134,-86.841220,35055


In [27]:
#extract zipcode from census data csv file

census_df=pd.read_csv("source_data/ACSDT5Y2024.B19013-Data.csv") # andrew: make sure the dataset is within the same folder as the code
census_df.head()


,GEO_ID,NAME,B19013_001E,B19013_001M,Unnamed: 4
0,Geography,Geographic Area Name,Estimate!!Median household income in the past ...,Margin of Error!!Median household income in th...,NaN
1,860Z200US00601,ZCTA5 00601,19454,1546,NaN
2,860Z200US00602,ZCTA5 00602,21420,1811,NaN
3,860Z200US00603,ZCTA5 00603,20933,1650,NaN
4,860Z200US00606,ZCTA5 00606,20992,3212,NaN


In [28]:
#let's get rid of every column besides the GEO_ID and the Median Household Income, since we won't be needing those!
census_df=census_df.drop(["B19013_001M","NAME", "Unnamed: 4"], axis=1)
census_df=census_df.iloc[1:]

In [29]:
#now, let's extract the zipcode!
census_df["zipcode"]=census_df["GEO_ID"].str[-5:]
census_df.head()

,GEO_ID,B19013_001E,zipcode
1,860Z200US00601,19454,00601
2,860Z200US00602,21420,00602
3,860Z200US00603,20933,00603
4,860Z200US00606,20992,00606
5,860Z200US00610,24496,00610


In [30]:
merged_data=pd.merge(df,census_df, on='zipcode', how='inner')

In [31]:
merged_data["median household income"]=merged_data["B19013_001E"]
merged_data=merged_data.drop(["B19013_001E"],axis=1)
merged_data.head()

,state,location,address,latitude,longitude,zipcode,GEO_ID,median household income
0,Alabama,Auburn,"346 W Magnolia Ave Auburn, AL 36832 US",32.606813,-85.487328,36832,860Z200US36832,42717
1,Alabama,Birmingham,"300 20th St S Birmingham, AL 35233 US",33.509721,-86.802756,35233,860Z200US35233,60336
2,Alabama,Birmingham,"3220 Morrow Rd Birmingham, AL 35235 US",33.595581,-86.647437,35235,860Z200US35235,63295
3,Alabama,Birmingham,"4719 Highway 280 Birmingham, AL 35242 US",33.422582,-86.698279,35242,860Z200US35242,117047
4,Alabama,Cullman,"1821 Cherokee Ave SW Cullman, AL 35055 US",34.154134,-86.841220,35055,860Z200US35055,58405


In [ ]:
merged_data.to_csv("outputs/merged_data_chipotle.csv", index=False)